In [1]:
## Part 1: CNN Architecture

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        # Conv2D with 16 filters, kernel size 3×3, stride 1, padding 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        # MaxPooling2D with kernel size 2×2, stride 2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Conv2D with 32 filters, kernel size 3×3, stride 1, padding 1
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        # Fully connected layer with 100 units (adjusted for 32×32 input)
        self.fc1 = nn.Linear(32 * 8 * 8, 100)
        # Fully connected layer with 10 units
        self.fc2 = nn.Linear(100, 10)
        
        self.relu = nn.ReLU()

    def forward(self, x):
        # Conv1 -> ReLU -> MaxPool
        x = self.pool(self.relu(self.conv1(x)))
        # Conv2 -> ReLU -> MaxPool
        x = self.pool(self.relu(self.conv2(x)))
        # Flatten
        x = x.view(x.size(0), -1)
        # FC1 -> ReLU
        x = self.relu(self.fc1(x))
        # FC2
        x = self.fc2(x)
        return x

# Test with 32×32 input
model = CNN()
sample_input = torch.randn(1, 3, 32, 32)
output = model(sample_input)
print(f"Input shape: {sample_input.shape}")
print(f"Output shape: {output.shape}")
print(model)

Input shape: torch.Size([1, 3, 32, 32])
Output shape: torch.Size([1, 10])
CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2048, out_features=100, bias=True)
  (fc2): Linear(in_features=100, out_features=10, bias=True)
  (relu): ReLU()
)


In [4]:
## Part 2: Model Deployment

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim

# Check which device is available
device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
)
print(f"Using device: {device}")

BATCH_SIZE = 32

# Prepare the Data
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 5
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {running_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "app/cifar10_model.pth")
print("Model saved!")

Using device: mps


100%|█████████████████████████████████████████| 170M/170M [23:55<00:00, 119kB/s]


Epoch 1/5, Loss: 1.4945
Epoch 2/5, Loss: 1.1793
Epoch 3/5, Loss: 1.0426
Epoch 4/5, Loss: 0.9498
Epoch 5/5, Loss: 0.8781
Model saved!


In [5]:
# Evaluate on test set
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 65.50%
